## Toy "Fixed Ratio" Compaction: Recent Token Retention

### Note

This notebook tests a simple toy retention policy that keeps the most recent 50% of cached positions. It is used to validate KV cache mutation and post compaction continuation before implementing the paper-inspired token eviction scoring method.

The recent token policy causes degenerate generation in this example and is not intended as the final compaction method.

In [19]:
!pip install -q transformers accelerate

In [20]:
import torch
import inspect

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

In [21]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_NEW_TOKENS = 256

DEVICE = "cuda"
DTYPE = torch.bfloat16

In [22]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)

model.eval()

print("Model:", type(model).__name__)
print("Device:", model.device)
print("Dtype:", model.dtype)

print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("KV heads:", model.config.num_key_value_heads)
print("Hidden size:", model.config.hidden_size)

head_dim = (
    model.config.hidden_size
    // model.config.num_attention_heads
)

print("Head dimension:", head_dim)

print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model: Qwen2ForCausalLM
Device: cuda:0
Dtype: torch.bfloat16
Layers: 24
Attention heads: 14
KV heads: 2
Hidden size: 896
Head dimension: 64
EOS token: '<|im_end|>'
EOS token ID: 151645


In [23]:
def get_kv_cache_bytes(past_key_values):
    key_bytes = 0
    value_bytes = 0

    for layer in past_key_values.layers:
        key_bytes += (
            layer.keys.numel()
            * layer.keys.element_size()
        )

        value_bytes += (
            layer.values.numel()
            * layer.values.element_size()
        )

    return key_bytes, value_bytes


def bytes_to_mib(num_bytes):
    return num_bytes / (1024 ** 2)


def print_cache_stats(past_key_values):
    key_bytes, value_bytes = get_kv_cache_bytes(
        past_key_values
    )

    total_bytes = key_bytes + value_bytes

    print(
        "KV tokens:",
        past_key_values.get_seq_length()
    )

    print(
        f"Key cache:   {bytes_to_mib(key_bytes):.4f} MiB"
    )

    print(
        f"Value cache: {bytes_to_mib(value_bytes):.4f} MiB"
    )

    print(
        f"Total KV:    {bytes_to_mib(total_bytes):.4f} MiB"
    )

In [24]:
def timed_cuda_call(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()

    start.record()

    result = fn()

    end.record()

    torch.cuda.synchronize()

    elapsed_ms = start.elapsed_time(end)

    return result, elapsed_ms

In [25]:
def generate_from_cache(
    outputs,
    past_key_values,
    generated_ids,
    attention_mask,
    max_new_tokens=MAX_NEW_TOKENS,
):

    assert (
        generated_ids.shape[1]
        == past_key_values.get_seq_length()
    )

    assert (
        attention_mask.shape[1]
        == past_key_values.get_seq_length()
    )

    next_token = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1,
        keepdim=True,
    )

    generated_token_ids = []
    decode_times_ms = []

    finished = False

    for _ in range(max_new_tokens):

        # --------------------------------
        # 1. Add predicted token
        # --------------------------------

        generated_ids = torch.cat(
            [generated_ids, next_token],
            dim=-1,
        )

        generated_token_ids.append(
            next_token.item()
        )

        # --------------------------------
        # 2. Extend attention mask by one
        # --------------------------------

        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (attention_mask.shape[0], 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=-1,
        )

        # --------------------------------
        # 3. Process new token into cache
        # --------------------------------

        def decode_step():
            with torch.no_grad():
                return model(
                    input_ids=next_token,
                    attention_mask=attention_mask,
                    past_key_values=past_key_values,
                    use_cache=True,
                )

        outputs, step_ms = timed_cuda_call(
            decode_step
        )

        decode_times_ms.append(step_ms)

        past_key_values = outputs.past_key_values

        # --------------------------------
        # 4. Correctness invariants
        # --------------------------------

        assert (
            generated_ids.shape[1]
            == past_key_values.get_seq_length()
        )

        assert (
            attention_mask.shape[1]
            == past_key_values.get_seq_length()
        )

        # --------------------------------
        # 5. Natural turn termination
        # --------------------------------

        if (
            next_token.item()
            == tokenizer.eos_token_id
        ):
            finished = True
            break

        # --------------------------------
        # 6. Predict following token
        # --------------------------------

        next_token = torch.argmax(
            outputs.logits[:, -1, :],
            dim=-1,
            keepdim=True,
        )

    return {
        "generated_ids": generated_ids,
        "past_key_values": past_key_values,
        "attention_mask": attention_mask,
        "outputs": outputs,
        "generated_token_ids": generated_token_ids,
        "decode_times_ms": decode_times_ms,
        "finished": finished,
    }

In [26]:
t1_messages = [
    {
        "role": "user",
        "content": (
            "I'm planning a tea party today. "
            "The theme is lavender and the special dessert "
            "is lemon cake. Remember these details for me."
        ),
    }
]

t1_text = tokenizer.apply_chat_template(
    t1_messages,
    tokenize=False,
    add_generation_prompt=True,
)

t1_inputs = tokenizer(
    t1_text,
    return_tensors="pt",
).to(model.device)

generated_ids = (
    t1_inputs["input_ids"].clone()
)

attention_mask = (
    t1_inputs["attention_mask"].clone()
)

print("TURN 1")
print()
print(t1_text)

print(
    "\nPrompt tokens:",
    generated_ids.shape[1]
)

TURN 1

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant


Prompt tokens: 55


In [27]:
def run_t1_prefill():

    with torch.no_grad():

        return model(
            input_ids=t1_inputs["input_ids"],
            attention_mask=attention_mask,
            use_cache=True,
        )


outputs, t1_prefill_ms = timed_cuda_call(
    run_t1_prefill
)

past_key_values = outputs.past_key_values


assert (
    generated_ids.shape[1]
    == past_key_values.get_seq_length()
)

assert (
    attention_mask.shape[1]
    == past_key_values.get_seq_length()
)


print(
    f"Turn 1 prefill: "
    f"{t1_prefill_ms:.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

Turn 1 prefill: 33.764 ms

KV tokens: 55
Key cache:   0.3223 MiB
Value cache: 0.3223 MiB
Total KV:    0.6445 MiB


In [28]:
t1_result = generate_from_cache(
    outputs=outputs,
    past_key_values=past_key_values,
    generated_ids=generated_ids,
    attention_mask=attention_mask,
)

generated_ids = (
    t1_result["generated_ids"]
)

past_key_values = (
    t1_result["past_key_values"]
)

attention_mask = (
    t1_result["attention_mask"]
)

outputs = (
    t1_result["outputs"]
)

t1_generated = (
    t1_result["generated_token_ids"]
)

t1_decode_times = (
    t1_result["decode_times_ms"]
)

t1_finished = (
    t1_result["finished"]
)


print(
    "Naturally terminated:",
    t1_finished
)

print(
    "Generated tokens:",
    len(t1_generated)
)

print(
    f"Total decode time: "
    f"{sum(t1_decode_times):.3f} ms"
)

print(
    f"Mean decode/token: "
    f"{sum(t1_decode_times) / len(t1_decode_times):.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

print("\nConversation:")

print(
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=False,
    )
)


assert t1_finished, (
    "Turn 1 hit MAX_NEW_TOKENS before EOS. "
    "Do not continue to Turn 2."
)

Naturally terminated: True
Generated tokens: 123
Total decode time: 3427.011 ms
Mean decode/token: 27.862 ms

KV tokens: 178
Key cache:   1.0430 MiB
Value cache: 1.0430 MiB
Total KV:    2.0859 MiB

Conversation:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant
Great! Here are the details for your tea party and lemon cake:

**Tea Party Details:**
- Date: [Insert Date]
- Time: [Insert Time]
- Location: [Insert Location]
- Guests: [Insert Guests]
- Theme: Lavender and Lemon
- Dessert: Lemon cake

**Lemon Cake Details:**
- Ingredients: [List of ingredients]
- Preparation: [Instructions for making the lemon cake]
- Serving: [Instructions for serving the lemon cake]

Please let me know if you need any additional details or if you have any specific requests for the party.<|i

In [29]:
original_seq_len = past_key_values.get_seq_length()

retain_ratio = 0.5
retain_count = int(original_seq_len * retain_ratio)

start_idx = original_seq_len - retain_count

retained_indices = torch.arange(
    start_idx,
    original_seq_len,
    device=model.device
)

print("Original cache length:", original_seq_len)
print("Retain count:", retain_count)
print("Retained indices:", retained_indices)

Original cache length: 178
Retain count: 89
Retained indices: tensor([ 89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102,
        103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
        131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144,
        145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158,
        159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172,
        173, 174, 175, 176, 177], device='cuda:0')


In [30]:
for i, layer in enumerate(past_key_values.layers):
    layer.keys = layer.keys[:, :, retained_indices, :]
    layer.values = layer.values[:, :, retained_indices, :]

print("Compacted all layers.")

Compacted all layers.


In [31]:
retained_token_ids = generated_ids[:, retained_indices]

print(
    tokenizer.decode(
        retained_token_ids[0],
        skip_special_tokens=False,
    )
)

]
- Location: [Insert Location]
- Guests: [Insert Guests]
- Theme: Lavender and Lemon
- Dessert: Lemon cake

**Lemon Cake Details:**
- Ingredients: [List of ingredients]
- Preparation: [Instructions for making the lemon cake]
- Serving: [Instructions for serving the lemon cake]

Please let me know if you need any additional details or if you have any specific requests for the party.<|im_end|>


In [32]:
cache_lengths = [
    layer.keys.shape[2]
    for layer in past_key_values.layers
]

print("Unique cache lengths:", set(cache_lengths))
print("Compacted cache length:", past_key_values.get_seq_length())

assert set(cache_lengths) == {retain_count}
assert past_key_values.get_seq_length() == retain_count

Unique cache lengths: {89}
Compacted cache length: 89


In [33]:
print("Logical Turn 1 length:", original_seq_len)
print("Physical compacted cache length:", past_key_values.get_seq_length())

t2_user_text = "What theme and special dessert did I choose?"

t2_suffix = (
    "<|im_start|>user\n"
    + t2_user_text
    + "<|im_end|>\n"
    + "<|im_start|>assistant\n"
)

t2_inputs = tokenizer(
    t2_suffix,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

print("Turn 2 token count:", t2_inputs["input_ids"].shape[1])

Logical Turn 1 length: 178
Physical compacted cache length: 89
Turn 2 token count: 17


In [34]:
logical_position = original_seq_len

for i in range(t2_inputs["input_ids"].shape[1]):
    token = t2_inputs["input_ids"][:, i:i+1]

    attention_mask_compacted = torch.ones(
        (1, past_key_values.get_seq_length() + 1),
        dtype=torch.long,
        device=model.device,
    )

    position_id = torch.tensor(
        [[logical_position]],
        dtype=torch.long,
        device=model.device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=token,
            attention_mask=attention_mask_compacted,
            position_ids=position_id,
            past_key_values=past_key_values,
            use_cache=True,
        )

    past_key_values = outputs.past_key_values
    logical_position += 1

print("Cache after Turn 2 suffix:", past_key_values.get_seq_length())
print("Next logical position:", logical_position)

Cache after Turn 2 suffix: 106
Next logical position: 195


In [35]:
next_token = torch.argmax(
    outputs.logits[:, -1, :],
    dim=-1,
    keepdim=True,
)

generated_t2 = []

for _ in range(50):
    generated_t2.append(next_token.item())

    attention_mask_compacted = torch.ones(
        (1, past_key_values.get_seq_length() + 1),
        dtype=torch.long,
        device=model.device,
    )

    position_id = torch.tensor(
        [[logical_position]],
        dtype=torch.long,
        device=model.device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask_compacted,
            position_ids=position_id,
            past_key_values=past_key_values,
            use_cache=True,
        )

    past_key_values = outputs.past_key_values
    logical_position += 1

    if next_token.item() == tokenizer.eos_token_id:
        break

    next_token = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1,
        keepdim=True,
    )

In [36]:
t2_response = tokenizer.decode(
    generated_t2,
    skip_special_tokens=False
)

print(t2_response)

Please let me know if you have any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any any
